### 1. Load the data

In [1]:
from pathlib import Path

import pandas as pd


PROCESSED_DATA_DIR = Path("../data/processed")

community_assignments = pd.read_csv(
    PROCESSED_DATA_DIR / "community_assignments.csv"
)

clean_edges = pd.read_csv(
    PROCESSED_DATA_DIR / "clean_story_keyword_edges.csv"
)

print("Community assignments:", community_assignments.shape)
print("Clean keyword edges:", clean_edges.shape)

display(community_assignments.head())
display(clean_edges.head())

Community assignments: (1266, 3)
Clean keyword edges: (20720, 3)


,story_id,louvain_community,greedy_community
0,8193,18,0
1,2,0,17
2,16387,18,0
3,16410,1,2
4,8226,12,0


,story_id,keyword_id,keyword
0,1319,929,Buttern
1,1453,30,on cattle
2,15342,206,Etymologie
3,5811,5813,Herrenhaus
4,3616,29,witches


### 2. story ID validation

In [2]:
community_assignments["story_id"] = (
    community_assignments["story_id"].astype(str)
)

clean_edges["story_id"] = (
    clean_edges["story_id"].astype(str)
)

### 3. Join stories with their Louvain communities

In [3]:
community_keywords = clean_edges.merge(
    community_assignments[
        ["story_id", "louvain_community"]
    ],
    on="story_id",
    how="inner"
)

print("Joined rows:", len(community_keywords))
display(community_keywords.head())

Joined rows: 5783


,story_id,keyword_id,keyword,louvain_community
0,15342,206,Etymologie,9
1,9307,4,historical legends,9
2,10498,354,Berg,10
3,17765,41,local legends,18
4,822,824,weiblich: Deichsel,16


### 4. Count keywords inside each community

In [4]:
keyword_counts = (
    community_keywords
    .groupby(
        ["louvain_community", "keyword"]
    )["story_id"]
    .nunique()
    .reset_index(name="story_count")
)

keyword_counts = keyword_counts.sort_values(
    ["louvain_community", "story_count"],
    ascending=[True, False]
)

display(keyword_counts.head(20))

,louvain_community,keyword,story_count
0,0,Danish people,12
1,0,Däne,12
2,0,"Sage, historische",12
3,0,historical legends,12
10,1,Franzose,96
14,1,"Sage, historische",96
18,1,historical legends,96
11,1,French people,95
17,1,from the french period,2
4,1,"30-/100-year war, blending",1


### 5. Select the top 10 keywords per community

In [5]:
TOP_KEYWORDS = 10

top_keywords = (
    keyword_counts
    .groupby("louvain_community")
    .head(TOP_KEYWORDS)
    .reset_index(drop=True)
)

display(top_keywords)

,louvain_community,keyword,story_count
0,0,Danish people,12
1,0,Däne,12
2,0,"Sage, historische",12
3,0,historical legends,12
4,1,Franzose,96
...,...,...,...
154,18,Fachliteratur,26
155,18,destruction,26
156,18,goldene Wiege,26
157,18,Untergang: Lit.,25


### 6. Print the results clearly

In [6]:
for community_id, group in top_keywords.groupby(
    "louvain_community"
):
    print(f"\nCommunity {community_id}")

    for _, row in group.iterrows():
        print(
            f"- {row['keyword']}: "
            f"{row['story_count']} stories"
        )


Community 0
- Danish people: 12 stories
- Däne: 12 stories
- Sage, historische: 12 stories
- historical legends: 12 stories

Community 1
- Franzose: 96 stories
- Sage, historische: 96 stories
- historical legends: 96 stories
- French people: 95 stories
- from the french period: 2 stories
- 30-/100-year war, blending: 1 stories
- Brücke: 1 stories
- Carwitz-Brücke: 1 stories
- Carwitz-bridge: 1 stories
- Danish people: 1 stories

Community 2
- Räuber: 31 stories
- Störtebecker: 31 stories
- robbers: 31 stories
- Störtebecker - Gödeke Michel: 30 stories
- Räubersage: 1 stories

Community 3
- Lokalsage: 62 stories
- Raubritter: 62 stories
- local legends: varia: 62 stories
- Robber-knight: 41 stories
- Robber-knight gen.: 21 stories
- Berg: 1 stories
- Etymologie: 1 stories
- Mountain open: 1 stories
- local legends: churches and passages: 1 stories

Community 4
- Sage, historische: 20 stories
- from the french period: 20 stories
- historical legends: 20 stories

Community 5
- Wasser: 56

### 7. Add community sizes and percentages

In [7]:
community_sizes = (
    community_assignments
    .groupby("louvain_community")["story_id"]
    .nunique()
    .rename("community_size")
)

top_keywords = top_keywords.merge(
    community_sizes,
    on="louvain_community",
    how="left"
)

top_keywords["percentage"] = (
    top_keywords["story_count"]
    / top_keywords["community_size"]
    * 100
).round(2)

display(top_keywords)

,louvain_community,keyword,story_count,community_size,percentage
0,0,Danish people,12,12,100.00
1,0,Däne,12,12,100.00
2,0,"Sage, historische",12,12,100.00
3,0,historical legends,12,12,100.00
4,1,Franzose,96,96,100.00
...,...,...,...,...,...
154,18,Fachliteratur,26,252,10.32
155,18,destruction,26,252,10.32
156,18,goldene Wiege,26,252,10.32
157,18,Untergang: Lit.,25,252,9.92


In [8]:
for community_id, group in top_keywords.groupby(
    "louvain_community"
):
    community_size = group["community_size"].iloc[0]

    print(
        f"\nCommunity {community_id} "
        f"({community_size} stories)"
    )

    for _, row in group.iterrows():
        print(
            f"- {row['keyword']}: "
            f"{row['story_count']} stories "
            f"({row['percentage']}%)"
        )


Community 0 (12 stories)
- Danish people: 12 stories (100.0%)
- Däne: 12 stories (100.0%)
- Sage, historische: 12 stories (100.0%)
- historical legends: 12 stories (100.0%)

Community 1 (96 stories)
- Franzose: 96 stories (100.0%)
- Sage, historische: 96 stories (100.0%)
- historical legends: 96 stories (100.0%)
- French people: 95 stories (98.96%)
- from the french period: 2 stories (2.08%)
- 30-/100-year war, blending: 1 stories (1.04%)
- Brücke: 1 stories (1.04%)
- Carwitz-Brücke: 1 stories (1.04%)
- Carwitz-bridge: 1 stories (1.04%)
- Danish people: 1 stories (1.04%)

Community 2 (31 stories)
- Räuber: 31 stories (100.0%)
- Störtebecker: 31 stories (100.0%)
- robbers: 31 stories (100.0%)
- Störtebecker - Gödeke Michel: 30 stories (96.77%)
- Räubersage: 1 stories (3.23%)

Community 3 (62 stories)
- Lokalsage: 62 stories (100.0%)
- Raubritter: 62 stories (100.0%)
- local legends: varia: 62 stories (100.0%)
- Robber-knight: 41 stories (66.13%)
- Robber-knight gen.: 21 stories (33.87%

### 8. Save the keyword summary

In [9]:
OUTPUT_PATH = (
    PROCESSED_DATA_DIR
    / "community_top_keywords.csv"
)

top_keywords.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved to:")
print(OUTPUT_PATH)

Saved to:
..\data\processed\community_top_keywords.csv


## Keyword interpretation conclusion

The cleaned story-keyword relationships were joined with the Louvain community
assignments.

For every community, the ten keywords occurring in the largest number of
stories were identified. The percentage of stories containing each keyword was
also calculated.

These keyword profiles provide an initial interpretation of the themes
represented by each detected community.